# Brooks PreciseFlex PF400 / PF3400

The PreciseFlex PF400 and PF3400 are SCARA robotic arms from Brooks Automation. They support:

- [Arms](../../capabilities/arms) (pick/place, Cartesian and joint movement, freedrive teaching)

The device communicates over Ethernet (TCP socket).

| Model | PLR Name | Rail | Notes |
|---|---|---|---|
| PreciseFlex 400 | `PreciseFlex400` | optional | 4-axis SCARA + gripper |
| PreciseFlex 3400 | `PreciseFlex3400Backend` | optional | Extended-reach variant (backend only) |

## Setup

In [ ]:
from pylabrobot.brooks import PreciseFlex400

pf = PreciseFlex400(host="192.168.0.1", port=10100, has_rail=False)
await pf.setup()

By default, `setup()` powers on the robot, attaches it, and homes it. Pass `skip_home=True` to the driver to skip homing.

## Arm capabilities

The PreciseFlex exposes an {class}`~pylabrobot.arms.orientable_arm.OrientableArm` on `pf.arm`. For the full arm API, see [Arms](../../capabilities/arms).

### Gripper control

In [ ]:
await pf.arm.open_gripper(gripper_width=120)
await pf.arm.close_gripper(gripper_width=80)
await pf.arm.backend.is_gripper_closed()

### Cartesian movement

Move the arm to a Cartesian location using {class}`~pylabrobot.brooks.precise_flex.PreciseFlexArmBackend.MoveToLocationParams` to set speed and elbow orientation:

In [ ]:
from pylabrobot.resources import Coordinate
from pylabrobot.brooks import PreciseFlexArmBackend

await pf.arm.move_to_location(
    location=Coordinate(x=290, y=659, z=100),
    direction=84.8,
    backend_params=PreciseFlexArmBackend.MoveToLocationParams(speed=50),
)

### Joint movement

Move the arm using joint coordinates with {class}`~pylabrobot.brooks.precise_flex.PreciseFlexArmBackend.MoveToJointPositionParams`:

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base or nearby equipment. Verify coordinates carefully.
```

In [ ]:
from pylabrobot.brooks.precise_flex import PFAxis

position = {
    PFAxis.BASE: 99.981,
    PFAxis.SHOULDER: -36.206,
    PFAxis.ELBOW: 83.063,
    PFAxis.WRIST: -331.7,
    PFAxis.GRIPPER: 126.084,
    PFAxis.RAIL: 0.0,
}
await pf.arm.move_to_joint_position(
    position,
    backend_params=PreciseFlexArmBackend.MoveToJointPositionParams(speed=30),
)

### Querying position

Get the current joint angles or Cartesian end-effector position:

In [ ]:
await pf.arm.request_joint_position()

In [ ]:
await pf.arm.request_gripper_location()

### Plate pick and place

Pick up and place plates using Cartesian coordinates. Use {class}`~pylabrobot.brooks.precise_flex.PreciseFlexArmBackend.PickUpParams` to configure the access pattern, grasp force, and elbow orientation, and {class}`~pylabrobot.brooks.precise_flex.PreciseFlexArmBackend.DropParams` for placement.

In [ ]:
from pylabrobot.brooks.precise_flex import VerticalAccess, HorizontalAccess

# Pick up a plate with vertical approach (default)
await pf.arm.pick_up_at_location(
    location=Coordinate(x=650.74, y=-345.922, z=5.05),
    direction=-9.921,
    resource_width=125,
)

# Place the plate
await pf.arm.drop_at_location(
    location=Coordinate(x=650.74, y=-345.922, z=5.05),
    direction=-9.921,
    resource_width=125,
)

For hotel-style (horizontal) access, pass a `HorizontalAccess` pattern via backend params:

In [ ]:
await pf.arm.pick_up_at_location(
    location=Coordinate(x=650.74, y=-345.922, z=5.05),
    direction=-9.921,
    resource_width=125,
    backend_params=PreciseFlexArmBackend.PickUpParams(
        access=HorizontalAccess(
            approach_distance_mm=50,
            clearance_mm=50,
            lift_height_mm=100,
        ),
        finger_speed_percent=40.0,
        grasp_force=8.0,
    ),
)

### Freedrive (teaching mode)

Enter freedrive mode to manually position the arm, then read coordinates for use in your protocol:

In [ ]:
await pf.arm.start_freedrive_mode(free_axes=[0])  # 0 = all axes

# Manually move the arm to the desired position, then read it:
await pf.arm.request_gripper_location()

await pf.arm.stop_freedrive_mode()

### Miscellaneous commands

Home the arm, move to the safe position, or halt all motion:

In [ ]:
await pf.arm.backend.move_to_safe()
await pf.arm.backend.halt()
await pf.driver.home()

## Teardown

In [ ]:
await pf.stop()